# Resample, smooth, compare and convert trajectories

**Question.** How do I resample, smooth, compare, convert and store trajectories, and how do I put them in the right time zone?

**You will get** trajectories resampled to a regular interval, a position at a chosen time, a smoothed track, a distance between two tracks, local-time timestamps, H3 grid cells, a spatial join to a grid, a GeoDataFrame, and a file round trip.

**Prerequisites.** [Start with a trajectory](analyze-a-simple-trajectory.md). The trajectory table in this notebook is a `TrajDataFrame`, the library's typed trajectory wrapper; every function also accepts a plain pandas or Polars dataframe.

**Data.** [GeoLife](https://www.microsoft.com/en-us/download/details.aspx?id=52367), the first 5 trips of users 000 and 001 (about 20,500 fixes, about 2 MB of the 313 MB archive), read with `fastmob.data.load_geolife`. Timestamps are UTC. The grid used for the spatial join is synthetic and built in the notebook. The notebook stops if the download fails.

> **Requires `fastmob` 0.3.4 or newer**, which provides the dataset loaders `fastmob.data.load_geolife` and `load_brightkite` (0.3.3 has them but caches GeoLife files across calls).

In [1]:
import time
from contextlib import contextmanager

_NOTEBOOK_START = time.perf_counter()
PHASES = {}


@contextmanager
def phase(name):
    """Add the wall-clock time of the block to PHASES[name]."""
    start = time.perf_counter()
    try:
        yield
    finally:
        PHASES[name] = PHASES.get(name, 0.0) + time.perf_counter() - start

In [2]:
with phase("setup"):
    get_ipython().run_line_magic("pip", "install -q 'fastmob>=0.3.4' pandas matplotlib geopandas shapely")

In [3]:
import fastmob.data

assert hasattr(fastmob.data, "load_geolife") and hasattr(fastmob.data, "load_brightkite"), (
    "This notebook needs fastmob 0.3.4 or newer (older releases lack the dataset loaders or have a loader caching bug)."
)

In [4]:
import tempfile
from pathlib import Path

cache_status = "cached (files from an earlier run were reused)" if (Path(tempfile.gettempdir()) / "fastmob_data" / "geolife").exists() else "cold (nothing cached before this run)"

## Load and inspect

`load_geolife` returns a `TrajDataFrame`. `.to_pandas()` gives an ordinary dataframe when you need one.

In [5]:
import fastmob
import numpy as np
import pandas as pd
from fastmob.data import load_geolife
from fastmob.trajectory import interpolate, interpolate_at, smooth, trajectory_distance

with phase("download and load"):
    tdf = load_geolife(users=["000", "001"], max_trips_per_user=5)

with phase("analysis"):
    fixes = tdf.to_pandas()
print(len(fixes), "fixes;", fixes["uid"].nunique(), "users")
fixes.head(3)

20500 fixes; 2 users


,uid,lat,lng,datetime
0,000,39.984702,116.318417,2008-10-23 02:53:04
1,000,39.984683,116.318450,2008-10-23 02:53:10
2,000,39.984686,116.318417,2008-10-23 02:53:15


## Put timestamps in local time

GeoLife is recorded in UTC but collected in Beijing (UTC+8). Anything that depends on the hour of the day, such as home and work or daily patterns, must use local time. `timezone_conversion` converts the `datetime` column in place from one time zone to another; the result has no time-zone information attached (it is tz-naive).

In [6]:
with phase("analysis"):
    local = fastmob.TrajDataFrame(fixes.copy())
    local.timezone_conversion(from_timezone="UTC", to_timezone="Asia/Shanghai")

pd.concat([fixes["datetime"].head(2).rename("utc"), local.to_pandas()["datetime"].head(2).rename("beijing")], axis=1)

,utc,beijing
0,2008-10-23 02:53:04,2008-10-23 10:53:04
1,2008-10-23 02:53:10,2008-10-23 10:53:10


The Beijing column is eight hours ahead of UTC: a fix at 02:53 UTC happened at 10:53 local time. The conversion changes the table in place and drops the time-zone label, so do not run it twice on the same table.

## Keep the data in time order

Most measures sort internally, but `sort_by_uid_and_datetime` returns a table ordered by user and then time, which is what you want before writing or exporting a table.

In [7]:
with phase("analysis"):
    ordered = tdf.sort_by_uid_and_datetime()
type(ordered).__name__, len(ordered)

('TrajDataFrame', 20500)

The table has the same 20,500 rows, now ordered by user and time.

## Resample to a regular interval

GPS recording is irregular. `interpolate` inserts positions on a regular time grid, here one every 60 seconds, by linear interpolation between neighbouring fixes. The same operation is available as the function `interpolate` and as the method `TrajDataFrame.interpolate`.

In [8]:
with phase("analysis"):
    regular = interpolate(fixes, method="linear", sampling_rate_s=60)
    regular_m = tdf.interpolate(method="linear", sampling_rate_s=60)

print(len(fixes), "fixes ->", len(regular), "positions at 60 s")
regular.head(3)

20500 fixes -> 20589 positions at 60 s


,uid,datetime,lat,lng
0,000,2008-10-23 02:53:04,39.984702,116.318417
1,000,2008-10-23 02:53:10,39.984683,116.318450
2,000,2008-10-23 02:53:15,39.984686,116.318417


Only 89 positions were added (20,500 to 20,589). GeoLife records every 1 to 5 seconds, so a 60-second grid mostly adds points inside the few longer gaps, and the first rows are the original fixes. A resampling interval much longer than the recording interval would instead reduce the density; choose it from what your analysis needs.

## Ask for a position at a given time

`interpolate_at` returns each user's estimated position at the timestamps you list, with a `valid` flag that is `False` when the user was not recorded at that time. Pass a list of timestamps (strings or `datetime64` values).

In [9]:
with phase("analysis"):
    at_time = interpolate_at(fixes, at=["2008-10-23 03:00:00"])
    at_time_m = tdf.interpolate_at(["2008-10-23 03:00:00"])

at_time

,uid,query_time,lat,lng,valid
0,000,2008-10-23 03:00:00,39.984356,116.304909,True
1,001,2008-10-23 03:00:00,NaN,NaN,False


User 000 has an estimated position at 03:00 UTC on 23 October 2008 (`valid` is `True`). User 001 has no estimate (`NaN`, `valid` is `False`), because that time lies outside what was recorded for this user. Always filter on `valid` before using the coordinates.

## Smooth GPS noise

`smooth` replaces each position with a Kalman-filter estimate at the same timestamp, so the number of rows does not change. `process_noise_std_km` sets how much the track may deviate from straight-line motion: larger values smooth less.

In [10]:
with phase("analysis"):
    smoothed = smooth(fixes, method="kalman_cv")
    smoothed_m = tdf.smooth()

shift_m = np.hypot(smoothed["lat"] - fixes["lat"], (smoothed["lng"] - fixes["lng"]) * np.cos(np.radians(fixes["lat"]))) * 111_320
print(f"median shift {shift_m.median():.1f} m, 95th percentile {shift_m.quantile(.95):.1f} m")

median shift 0.3 m, 95th percentile 1.5 m


The filter moves the median point by about 0.3 m and 95% of points by less than 1.5 m. That is light denoising: consumer GPS error is several metres, so the result stays close to the measurements. Increase the noise setting to smooth less, decrease it to smooth more, and check the effect on the measure you care about.

## Compare two trajectories

`trajectory_distance` measures how different two tracks are. The default `dtw` (dynamic time warping) lets the two tracks run at different speeds and returns a distance in kilometres summed over the best alignment. It compares exactly two single tracks, so slice one user each. We use the first 300 fixes of each.

In [11]:
with phase("analysis"):
    first, second = fixes[fixes["uid"] == "000"].head(300), fixes[fixes["uid"] == "001"].head(300)
    distance = trajectory_distance(first, second, method="dtw")
    distance_m = fastmob.TrajDataFrame(first).trajectory_distance(fastmob.TrajDataFrame(second), method="dtw")

print(f"DTW distance between user 000 and user 001: {distance:.0f} km")

DTW distance between user 000 and user 001: 496 km


The distance is about 496 km summed over the best alignment of 300 fixes against 300 fixes, an average of about 1.7 km between aligned points: the two users were not moving along the same route. The value grows with the number of points, so compare tracks of similar length.

## Summaries from the same object

Several measures are also methods of the `TrajDataFrame`, so a quick summary needs no imports: `jump_lengths`, `radius_of_gyration`, `compress`, `stay_locations` and `work_location`. They are the same functions used in the earlier notebooks.

In [12]:
with phase("analysis"):
    radius = tdf.radius_of_gyration()
    jumps = tdf.jump_lengths()
    compressed = tdf.compress(spatial_radius_km=0.2)
    stops = tdf.stay_locations(minutes_for_a_stop=20, spatial_radius_km=0.2)
    work = tdf.work_location()

print(len(compressed), "fixes after compress;", len(stops), "stops")
radius.to_pandas()

629 fixes after compress; 32 stops


,uid,radius_of_gyration
0,000,5.479766
1,001,6.320922


The methods return the same results as the functions: compression leaves 629 of 20,500 fixes, 32 stops are found, and the radius of gyration is about 5.5 km for user 000 and 6.3 km for user 001 (in kilometres). Measures that return one row per user come back as an Arrow table; `.to_pandas()` converts it.

## Move between tables

`settings_from` copies the column settings of one `TrajDataFrame` to another (useful after building a new table from an old one). `to_geodataframe` converts to a GeoPandas table with point geometry, which needs `geopandas`; it is installed by the first cell of this notebook.

In [13]:
with phase("analysis"):
    clone = fastmob.TrajDataFrame(fixes.copy())
    clone.settings_from(tdf)
    gdf = tdf.to_geodataframe()

print(type(gdf).__name__, gdf.crs, len(gdf))
gdf.head(2)

GeoDataFrame EPSG:4326 20500


,uid,lat,lng,datetime,geometry
0,000,39.984702,116.318417,2008-10-23 02:53:04,POINT (116.31842 39.9847)
1,000,39.984683,116.318450,2008-10-23 02:53:10,POINT (116.31845 39.98468)


The GeoDataFrame has one point per fix in longitude and latitude (EPSG:4326), so it can be used with any GeoPandas tool.

## Snap positions to a grid

`mapping` assigns each fix to the polygon (tile) that contains it, using a table of polygons with a `tile_id` column. We build a 5 by 5 grid over the extent of the data to keep the example self-contained; use your own zones (census tracts, districts) in practice.

In [14]:
import geopandas as gpd
from shapely.geometry import box

lat0, lat1, lng0, lng1 = fixes["lat"].min(), fixes["lat"].max(), fixes["lng"].min(), fixes["lng"].max()
dlat, dlng = (lat1 - lat0) / 5, (lng1 - lng0) / 5
cells = [box(lng0 + j * dlng, lat0 + i * dlat, lng0 + (j + 1) * dlng, lat0 + (i + 1) * dlat) for i in range(5) for j in range(5)]
tessellation = gpd.GeoDataFrame({"tile_id": range(25)}, geometry=cells, crs="EPSG:4326")

with phase("analysis"):
    mapped = tdf.mapping(tessellation, remove_na=True)

mapped.to_pandas().head(3)

,uid,lat,lng,datetime,tile_id
0,000,39.984702,116.318417,2008-10-23 02:53:04,13
1,000,39.984683,116.318450,2008-10-23 02:53:10,13
2,000,39.984686,116.318417,2008-10-23 02:53:15,13


Each fix now has the `tile_id` of the grid cell that contains it (the first fixes fall in tile 13). The grid is synthetic: its tile ids mean nothing outside this example.

## H3 cells

H3 is a global hexagonal grid. `latlng_to_h3` adds a cell id to every row at a chosen resolution (9 is a cell roughly 200 m across) and `h3_to_latlng` converts cell ids back to the cell centres. Use it to count visits per place without defining zones.

In [15]:
with phase("analysis"):
    cells_df = fastmob.latlng_to_h3(fixes, resolution=9)
    centres = fastmob.h3_to_latlng(cells_df.drop_duplicates("h3_cell")[["h3_cell"]])

print(cells_df["h3_cell"].nunique(), "distinct cells at resolution 9")
centres.head(3)

340 distinct cells at resolution 9


,h3_cell,center_lat,center_lng
0,617866870095347711,39.984411,116.318052
13,617866870096920575,39.984443,116.314793
34,617866869875933183,39.984475,116.311534


The 20,500 fixes fall in 340 distinct cells at resolution 9. `h3_to_latlng` gives each cell's centre, which is a convenient label for a place; the centre is not a visited position.

## Store and reload

`io.write` saves a table to CSV (or another supported format, chosen by the file extension) and `io.read` loads it back. `io.load_geolife_trajectories` reads GeoLife `.plt` files from a folder, which is what `load_geolife` calls after downloading.

In [16]:
import tempfile
from pathlib import Path

from fastmob import io

with phase("analysis"):
    path = Path(tempfile.mkdtemp()) / "geolife_subset.csv"
    io.write(tdf, path)
    reloaded = io.read(path)
    from_plt = io.load_geolife_trajectories(Path(tempfile.gettempdir()) / "fastmob_data" / "geolife", user_ids=["000"])

print(len(reloaded), "rows reloaded;", len(from_plt), "rows read from the cached .plt files of user 000")

20500 rows reloaded; 3424 rows read from the cached .plt files of user 000


The round trip returns all 20,500 rows. Reading the cached `.plt` files of user 000 directly returns 3,424 rows, the five trips that `load_geolife` extracted for that user.

## Which datasets does the library list?

`list_datasets` and `get_dataset_info` describe the datasets of the scikit-mobility collection (name, type, source, size).

In [17]:
from fastmob.data import get_dataset_info, list_datasets

names = list_datasets()
print(names)
{k: get_dataset_info("foursquare_nyc")[k] for k in ("name", "data_type", "download_format")}

['flow_foursquare_nyc', 'foursquare_nyc', 'nyc_boundaries', 'parking_san_francisco', 'taxi_san_francisco']


{'name': 'Foursquare_NYC', 'data_type': 'trajectory', 'download_format': 'zip'}

Five datasets are listed. `foursquare_nyc` is a trajectory dataset distributed as a zip file. Only the metadata is read here; the data themselves are not downloaded.

## Interpretation and limits

- Resampling creates positions that were never observed. Interpolated points are suitable for plots and for time-aligned comparison, not as additional evidence of where someone was.
- `smooth` replaces measured positions with model estimates; the filter assumes roughly constant velocity and handles sharp turns and stops poorly.
- Dynamic time warping compares shapes; it ignores clock time and the scale is in kilometres summed along the alignment, so it grows with track length.
- `mapping` gives a tile id only to fixes inside a polygon; with `remove_na=True` fixes outside the grid are dropped.
- `load_dataset` (the scikit-mobility download) is not demonstrated: its source host did not resolve when this notebook was prepared, so there is no executed example of it.

## Next steps

- Build stops, trips and tours from these fixes: [Build the trajectory hierarchy](trajectory-hierarchy.md).
- Count flows between zones: [Population measures and flows](population-measures.md).

## Related API

[`interpolate`](https://gefgu.github.io/fastmob/reference/trajectory/#fastmob.trajectory.interpolate) ·
[`interpolate_at`](https://gefgu.github.io/fastmob/reference/trajectory/#fastmob.trajectory.interpolate_at) ·
[`smooth`](https://gefgu.github.io/fastmob/reference/trajectory/#fastmob.trajectory.smooth) ·
[`trajectory_distance`](https://gefgu.github.io/fastmob/reference/trajectory/#fastmob.trajectory.trajectory_distance) ·
[`timezone_conversion`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.timezone_conversion) ·
[`sort_by_uid_and_datetime`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.sort_by_uid_and_datetime) ·
[`interpolate`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.interpolate) ·
[`interpolate_at`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.interpolate_at) ·
[`smooth`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.smooth) ·
[`trajectory_distance`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.trajectory_distance) ·
[`jump_lengths`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.jump_lengths) ·
[`radius_of_gyration`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.radius_of_gyration) ·
[`compress`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.compress) ·
[`stay_locations`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.stay_locations) ·
[`work_location`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.work_location) ·
[`settings_from`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.settings_from) ·
[`to_geodataframe`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.to_geodataframe) ·
[`mapping`](https://gefgu.github.io/fastmob/reference/data-structures/traj-dataframe/#fastmob.core.trajectory_dataframe.TrajDataFrame.mapping) ·
[`latlng_to_h3`](https://gefgu.github.io/fastmob/reference/preprocessing/#fastmob.preprocessing.latlng_to_h3) ·
[`h3_to_latlng`](https://gefgu.github.io/fastmob/reference/preprocessing/#fastmob.preprocessing.h3_to_latlng) ·
[`read`](https://gefgu.github.io/fastmob/reference/io/#fastmob.io.file.read) ·
[`write`](https://gefgu.github.io/fastmob/reference/io/#fastmob.io.file.write) ·
[`load_geolife_trajectories`](https://gefgu.github.io/fastmob/reference/io/#fastmob.io.file.load_geolife_trajectories) ·
[`list_datasets`](https://gefgu.github.io/fastmob/reference/data/#fastmob.data.list_datasets) ·
[`get_dataset_info`](https://gefgu.github.io/fastmob/reference/data/#fastmob.data.get_dataset_info)

## Run time

Measured by the notebook's own wall clock.

In [18]:
import datetime
import platform
from importlib.metadata import version

import pandas as pd

total = time.perf_counter() - _NOTEBOOK_START
measured = sum(PHASES.values())
timing = pd.DataFrame({"seconds": {**PHASES, "untimed cells and display": total - measured, "total notebook runtime": total}}).round(2)
print(timing.to_string())
print()
print("Download:", 'about 2 MB of the 313 MB GeoLife archive (HTTP range requests)')
print("Cache:", cache_status)
print("Rows analysed:", len(fixes))
print("Environment: Python", platform.python_version(), "| fastmob", version("fastmob"), "| pandas", version("pandas"),
      "| measured", datetime.datetime.now(datetime.timezone.utc).date().isoformat())

                           seconds
setup                        27.68
download and load             2.03
analysis                     11.89
untimed cells and display    13.24
total notebook runtime       54.84

Download: about 2 MB of the 313 MB GeoLife archive (HTTP range requests)
Cache: cold (nothing cached before this run)
Rows analysed: 20500


Environment: Python 3.10.12 | fastmob 0.3.4 | pandas 2.3.3 | measured 2026-10-27
